In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(r"UScomments.csv", on_bad_lines = "skip")

In [ ]:
df.head(5)

In [ ]:
type(df)

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

### Sentiment Analysis

In [ ]:
import nltk

In [ ]:
nltk.download("vader_lexicon")

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
sia = SentimentIntensityAnalyzer()

In [ ]:
df['comment_text']

In [ ]:
def analyze_comment_sentiment(comment):
    compound = sia.polarity_scores(comment)['compound']

    if compound > 0.05:
        label = 'Positive'
        insight = 'The comment express positive emotion or approval'

    elif compound <0.05:
        label = 'Negative'
        insight = 'The comment shows dis-satisfaction or negative emotion'

    else:
        label = 'Neutral'
        insight = 'The comment is informational or emotionallu neutral'

    return {
        'label' : label,
        'score': compound,
        'insight': insight
    }

In [ ]:
analyze_comment_sentiment('What a lovely day')

# Emoji's Analysis

In [ ]:
%pip install emoji

In [ ]:
import emoji

In [ ]:
df['comment_text'].head(20)

In [ ]:
comment = "trending 😉"

In [ ]:
emoji.EMOJI_DATA

In [ ]:
emoji_list = []

for char in comment:
    if char in emoji.EMOJI_DATA:
        emoji_list.append(char)

In [ ]:
emoji_list

In [ ]:
all_emojis_list = []

for comment in df['comment_text'].dropna():
    for char in comment:
        if char in emoji.EMOJI_DATA:
            all_emojis_list.append(char)

In [ ]:
all_emojis_list[0:10]

In [ ]:
len(all_emojis_list)

In [ ]:
from collections import Counter

In [ ]:
emojis_count_list = Counter(all_emojis_list).most_common(10)

In [ ]:
emojis_count_list

In [ ]:
emojis = [emoji for emoji, count in emojis_count_list]

In [ ]:
counts = [count for emoji, count in emojis_count_list]

In [ ]:
import plotly.express as px

In [ ]:
px.bar(x=emojis, y=counts, title = 'Most used emojis in the Youtube comments', labels = {'x':'Emoji', 'y':'Count'})

### Collecting Entire Data of Youtube

In [ ]:
import os

In [ ]:
files = os.listdir(r"C:\Users\PCL\Desktop\pbi\python_projects\additional_data")

In [ ]:
files

In [ ]:
files_csv = [file for file in files if '.csv' in file]

In [ ]:
files_csv

In [ ]:
full_df = pd.DataFrame()
path = r"C:\Users\PCL\Desktop\pbi\python_projects\additional_data"

for file in files_csv:
    current_df = pd.read_csv(path+'/'+file, encoding="iso=8859-1", on_bad_lines = "skip")
    full_df = pd.concat([current_df, full_df], ignore_index=True)

In [ ]:
full_df.shape

In [ ]:
full_df.head(5)

In [ ]:
full_df.columns

In [ ]:
full_df[full_df.duplicated()].shape

In [ ]:
full_df = full_df.drop_duplicates()

In [ ]:
full_df.shape

In [ ]:
full_df.to_csv(r'C:\Users\PCL\Desktop\pbi\python_projects\Whole_Youtube_data.csv', index = False)

In [ ]:
full_df.to_json(r'C:\Users\PCL\Desktop\pbi\python_projects\Whole_Youtube_data_sample.json')

In [ ]:
%pip install sqlalchemy

In [ ]:
from sqlalchemy import create_engine

In [ ]:
engine = create_engine(r'sqlite:///C:\Users\PCL\Desktop\pbi\python_projects\youtube_data.sqlite')

In [ ]:
full_df.to_sql('Users', con = engine, if_exists = 'replace')

### Which category dominates the Youtube ?

In [ ]:
full_df['trending_date']

In [ ]:
full_df['trending_date'] = pd.to_datetime(full_df['trending_date'], format = '%y.%d.%m')

In [ ]:
full_df.dtypes

In [ ]:
import json

In [ ]:
path = r'C:\Users\PCL\Desktop\pbi\python_projects\additional_data\US_category_id.json'

In [ ]:
with open(path, 'r', encoding = 'utf-8') as f:
    data = json.load(f)

In [ ]:
data

In [ ]:
data['items'][0]

In [ ]:
data['items'][0]['snippet']['title']

In [ ]:
cat_dict = {}

for item in data['items']:
    cat_dict[int(item['id'])]=item['snippet']['title']

In [ ]:
cat_dict

In [ ]:
full_df['category_name'] = full_df['category_id'].map(cat_dict)

In [ ]:
full_df.head(5)

In [ ]:
pivot_df = full_df.groupby(['trending_date', 'category_name'])['views'].sum().unstack(fill_value = 0)

In [ ]:
pivot_df.head(5)

In [ ]:
area_chart = px.area(
    data_frame = pivot_df,
    x=pivot_df.index,
    y=pivot_df.columns,
    title='Trending Momentum Over time by Category'
)

In [ ]:
area_chart

In [ ]:
top_categories = full_df.groupby(['category_name'])['views'].sum().nlargest(6).index

In [ ]:
top_categories

In [ ]:
pivot_df[top_categories]

In [ ]:
filtered_df=pivot_df[top_categories]

In [ ]:
area_chart = px.area(
    data_frame = pivot_df,
    x=filtered_df.index,
    y=filtered_df.columns,
    title='Trending Momentum Over time by Category (top 6)'
)

In [ ]:
area_chart

### Do viral videos actually get engagement?

In [ ]:
full_df['engagement_rate'] = (full_df['likes']+full_df['comment_count'])/full_df['views']

In [ ]:
bubble = px.scatter(full_df, 
           x = 'views', 
           y= 'engagement_rate', 
           size='comment_count', 
           color='category_name', 
           hover_name='title', 
           title='Engagement Bubble Map: Views vs Engagement Rate', 
           size_max= 60)

In [ ]:
bubble

In [ ]:
bubble.update_xaxes(type='log')

### Views vs Engagement: Inside Youtube's Algorithm

In [ ]:
full_df.columns

In [ ]:
category_metrics = full_df.groupby('category_name').agg(
    total_views = ('views', 'sum'),
    avg_engagement_efficiency = ('engagement_rate','mean'),
    video_count=('video_id', 'count')
).reset_index()


In [ ]:
category_metrics

In [ ]:
tree_map = px.treemap(
    category_metrics,
    path=['category_name'],
    values = 'total_views',
    color  = 'avg_engagement_efficiency',
    color_continuous_scale = 'RdYlGn',
    title = 'Category Attention Share with Enaggement Efficiency Overlay',
    hover_data={
        'total_views': ':,.0f',
        'avg_engagement_efficiency':':,.3f',
        'video_count': True
    }
)

In [ ]:
tree_map

### Is the audience actually engaged?

In [ ]:
full_df.columns

In [ ]:
full_df["engagement_rate"].describe()

In [ ]:
category_engagement_stats = full_df.groupby('category_name')['engagement_rate'].describe()

In [ ]:
category_engagement_stats.sort_values('mean', ascending=False)

In [ ]:
px.box(full_df, 
       x='category_name',
       y='engagement_rate',
       color='category_name',
       title='Audience Engagement by Category'
      )